# SAM 3 person tracking with a user-drawn polygon danger zone

This notebook is based on `tracking_clean_with_dangerzone.ipynb`. It lets you draw a polygon on a representative video frame, saves the polygon as normalized coordinates, and reports a danger-zone entry when a tracked person's foot point remains inside the polygon for several consecutive frames.

Polygon editor controls:
- Left click: add a vertex
- Right click or Backspace: undo the last vertex
- `R`: reset all vertices
- `Enter` or `S`: confirm and save
- `Esc`: cancel


In [ ]:
import glob
import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import sam3
import torch
from sam3.model_builder import build_sam3_video_predictor
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    render_masklet_frame,
    visualize_formatted_frame_output,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12

sam3_root = Path(sam3.__file__).resolve().parent.parent
gpus_to_use = range(torch.cuda.device_count())
predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)


## Polygon editor and configuration helpers

The editor uses a native OpenCV window. The saved points are normalized to the `[0, 1]` range, so the zone can be reused with another video from the same fixed camera even if its resolution changes.


In [ ]:
def load_video_frames(video_path):
    video_path = str(video_path)
    if video_path.lower().endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        return frames

    frames = glob.glob(os.path.join(video_path, "*.jpg"))
    try:
        frames.sort(key=lambda p: int(Path(p).stem))
    except ValueError:
        frames.sort()
    return frames


def get_video_fps(video_path, default_fps=10.0):
    video_path = str(video_path)
    if video_path.lower().endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if fps and fps > 0:
            return float(fps)
    return float(default_fps)


def select_polygon_on_frame(frame_rgb, max_display_width=1280):
    if frame_rgb is None or frame_rgb.size == 0:
        raise ValueError("The polygon editor received an empty frame.")

    height, width = frame_rgb.shape[:2]
    scale = min(1.0, max_display_width / width)
    display_size = (int(width * scale), int(height * scale))
    display_base = cv2.resize(
        cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR),
        display_size,
        interpolation=cv2.INTER_AREA,
    )
    points = []
    window_name = "Draw danger zone - Enter: save, R: reset, Esc: cancel"

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((x, y))
        elif event == cv2.EVENT_RBUTTONDOWN and points:
            points.pop()

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, *display_size)
    cv2.setMouseCallback(window_name, on_mouse)

    try:
        while True:
            canvas = display_base.copy()
            if len(points) >= 2:
                cv2.polylines(
                    canvas,
                    [np.asarray(points, dtype=np.int32)],
                    len(points) >= 3,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
            for index, point in enumerate(points):
                cv2.circle(canvas, point, 5, (0, 0, 255), -1, cv2.LINE_AA)
                cv2.putText(
                    canvas,
                    str(index + 1),
                    (point[0] + 7, point[1] - 7),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
            cv2.putText(
                canvas,
                "Left click:add  Right click/Backspace:undo  Enter/S:save  R:reset  Esc:cancel",
                (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )
            cv2.imshow(window_name, canvas)
            key = cv2.waitKey(20) & 0xFF

            if key in (13, 10, ord("s")):
                if len(points) < 3:
                    print("A polygon needs at least three points.")
                    continue
                contour = np.asarray(points, dtype=np.float32)
                if abs(cv2.contourArea(contour)) < 10:
                    print("The polygon is too small. Draw it again.")
                    continue
                break
            if key in (8, 127) and points:
                points.pop()
            elif key == ord("r"):
                points.clear()
            elif key == 27:
                raise KeyboardInterrupt("Polygon drawing was cancelled.")
    finally:
        cv2.destroyWindow(window_name)

    display_width, display_height = display_size
    return [
        [x / display_width, y / display_height]
        for x, y in points
    ]


def save_zone_config(config_path, normalized_points, source_video):
    config_path = Path(config_path)
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config = {
        "version": 1,
        "camera_mode": "fixed",
        "source_video": str(Path(source_video).resolve()),
        "points_normalized": normalized_points,
    }
    config_path.write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"Saved danger-zone config: {config_path.resolve()}")


def load_zone_config(config_path):
    config = json.loads(Path(config_path).read_text(encoding="utf-8"))
    points = config["points_normalized"]
    if len(points) < 3:
        raise ValueError("The saved danger zone has fewer than three points.")
    return points


def normalized_polygon_to_pixels(normalized_points, width, height):
    polygon = np.asarray(
        [
            [round(x * width), round(y * height)]
            for x, y in normalized_points
        ],
        dtype=np.int32,
    )
    polygon[:, 0] = np.clip(polygon[:, 0], 0, width - 1)
    polygon[:, 1] = np.clip(polygon[:, 1], 0, height - 1)
    return polygon


## Tracking and polygon intrusion detection

A person's floor-contact point is estimated from the bottom rows of the SAM mask. Positive `pointPolygonTest` distance means that point is inside the danger zone. A margin and consecutive-frame confirmation reduce flicker near the boundary.


In [ ]:
def propagate_in_video(predictor, session_id):
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request={
            "type": "propagate_in_video",
            "session_id": session_id,
        }
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]
    return outputs_per_frame


def get_foot_point(mask, bottom_band_height=5):
    mask = np.squeeze(mask).astype(bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    bottom_y = int(ys.max())
    bottom_xs = xs[ys >= bottom_y - (bottom_band_height - 1)]
    return int(np.median(bottom_xs)), bottom_y


def export_tracking_video_with_polygon_zone(
    video_frames,
    outputs_per_frame,
    normalized_polygon,
    output_path,
    fps,
    zone_margin=8.0,
    confirm_frames=3,
    exit_confirm_frames=3,
    mask_alpha=0.5,
    zone_alpha=0.18,
):
    if not video_frames:
        raise ValueError("No video frames were loaded.")
    if confirm_frames < 1 or exit_confirm_frames < 1:
        raise ValueError("Confirmation frame counts must be at least one.")

    first_frame = load_frame(video_frames[0])
    height, width = first_frame.shape[:2]
    polygon = normalized_polygon_to_pixels(
        normalized_polygon, width, height
    )
    contour = polygon.astype(np.float32).reshape((-1, 1, 2))
    object_states = {}
    intrusion_events = []

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not open output video: {output_path}")

    try:
        for frame_idx in sorted(outputs_per_frame):
            rgb_frame = load_frame(video_frames[frame_idx])
            frame_output = outputs_per_frame[frame_idx]
            rendered_frame = render_masklet_frame(
                rgb_frame,
                frame_output,
                frame_idx=frame_idx,
                alpha=mask_alpha,
            )

            active_intrusions = []
            for object_id, mask in zip(
                frame_output["out_obj_ids"],
                frame_output["out_binary_masks"],
            ):
                object_id = int(object_id)
                foot_point = get_foot_point(mask)
                if foot_point is None:
                    continue

                distance = cv2.pointPolygonTest(contour, foot_point, True)
                state = object_states.setdefault(
                    object_id,
                    {
                        "inside": False,
                        "inside_frames": 0,
                        "outside_frames": 0,
                    },
                )

                if distance > zone_margin:
                    state["inside_frames"] += 1
                    state["outside_frames"] = 0
                    if (
                        not state["inside"]
                        and state["inside_frames"] >= confirm_frames
                    ):
                        state["inside"] = True
                        event = {
                            "frame_idx": frame_idx,
                            "time_seconds": frame_idx / fps,
                            "object_id": object_id,
                            "foot_point": foot_point,
                        }
                        intrusion_events.append(event)
                        print(
                            f"[DANGER ZONE ENTRY] frame={frame_idx}, "
                            f"time={event['time_seconds']:.2f}s, "
                            f"person_id={object_id}"
                        )
                elif distance < -zone_margin:
                    state["outside_frames"] += 1
                    state["inside_frames"] = 0
                    if state["outside_frames"] >= exit_confirm_frames:
                        state["inside"] = False
                else:
                    state["inside_frames"] = 0
                    state["outside_frames"] = 0

                foot_color = (255, 0, 0) if state["inside"] else (0, 255, 0)
                cv2.circle(rendered_frame, foot_point, 7, foot_color, -1)
                if state["inside"]:
                    active_intrusions.append(object_id)

            zone_overlay = rendered_frame.copy()
            zone_color = (255, 0, 0) if active_intrusions else (255, 215, 0)
            cv2.fillPoly(zone_overlay, [polygon], zone_color)
            rendered_frame = cv2.addWeighted(
                zone_overlay, zone_alpha, rendered_frame, 1.0 - zone_alpha, 0
            )
            cv2.polylines(
                rendered_frame, [polygon], True, zone_color, 4, cv2.LINE_AA
            )
            label_position = tuple(polygon[np.argmin(polygon[:, 1])])
            cv2.putText(
                rendered_frame,
                "DANGER ZONE",
                (int(label_position[0]), max(35, int(label_position[1]) - 12)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                zone_color,
                2,
                cv2.LINE_AA,
            )

            if active_intrusions:
                warning = "DANGER! PERSON " + ", ".join(
                    map(str, sorted(active_intrusions))
                )
                cv2.rectangle(
                    rendered_frame, (15, 15), (width - 15, 75), (255, 0, 0), -1
                )
                cv2.putText(
                    rendered_frame,
                    warning,
                    (30, 58),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.1,
                    (255, 255, 255),
                    3,
                    cv2.LINE_AA,
                )

            writer.write(cv2.cvtColor(rendered_frame, cv2.COLOR_RGB2BGR))
    finally:
        writer.release()

    print(f"Saved tracking video to: {Path(output_path).resolve()}")
    print(f"Detected {len(intrusion_events)} danger-zone entries.")
    return intrusion_events


## Choose the video and draw or load its danger zone

Set `redraw_zone = True` whenever the camera position or desired zone changes. Otherwise the saved JSON configuration is reused. `zone_frame_idx` controls which representative frame is shown in the editor.


In [ ]:
video_path = sam3_root / "assets" / "videos" / "factory_2.mp4"
zone_config_path = sam3_root / "examples" / "danger_zone_factory_2.json"
output_video_path = (
    sam3_root / "examples" / "sam3_person_tracking_polygon_danger_zone.mp4"
)
zone_frame_idx = 0
redraw_zone = True

video_frames_for_vis = load_video_frames(video_path)
if not video_frames_for_vis:
    raise RuntimeError(f"No frames were loaded from: {video_path}")
if not 0 <= zone_frame_idx < len(video_frames_for_vis):
    raise IndexError("zone_frame_idx is outside the video frame range.")

if redraw_zone or not zone_config_path.exists():
    reference_frame = load_frame(video_frames_for_vis[zone_frame_idx])
    normalized_polygon = select_polygon_on_frame(reference_frame)
    save_zone_config(zone_config_path, normalized_polygon, video_path)
else:
    normalized_polygon = load_zone_config(zone_config_path)
    print(f"Loaded danger-zone config: {zone_config_path.resolve()}")

preview_frame = load_frame(video_frames_for_vis[zone_frame_idx]).copy()
preview_h, preview_w = preview_frame.shape[:2]
preview_polygon = normalized_polygon_to_pixels(
    normalized_polygon, preview_w, preview_h
)
preview_overlay = preview_frame.copy()
cv2.fillPoly(preview_overlay, [preview_polygon], (255, 215, 0))
preview_frame = cv2.addWeighted(preview_overlay, 0.2, preview_frame, 0.8, 0)
cv2.polylines(preview_frame, [preview_polygon], True, (255, 215, 0), 4)
plt.figure(figsize=(12, 7))
plt.imshow(preview_frame)
plt.title("Saved polygon danger zone")
plt.axis("off")
plt.show()


## Run SAM 3 person tracking


In [ ]:
response = predictor.handle_request(
    request={
        "type": "start_session",
        "resource_path": str(video_path),
    }
)
session_id = response["session_id"]

_ = predictor.handle_request(
    request={
        "type": "reset_session",
        "session_id": session_id,
    }
)

prompt_frame_idx = 0
response = predictor.handle_request(
    request={
        "type": "add_prompt",
        "session_id": session_id,
        "frame_index": prompt_frame_idx,
        "text": "person",
    }
)
prompt_output = response["outputs"]

plt.close("all")
visualize_formatted_frame_output(
    prompt_frame_idx,
    video_frames_for_vis,
    outputs_list=[
        prepare_masks_for_visualization({prompt_frame_idx: prompt_output})
    ],
    titles=["SAM 3 person prompt output"],
    figsize=(8, 5),
)


In [ ]:
outputs_per_frame = propagate_in_video(predictor, session_id)

intrusion_events = export_tracking_video_with_polygon_zone(
    video_frames=video_frames_for_vis,
    outputs_per_frame=outputs_per_frame,
    normalized_polygon=normalized_polygon,
    output_path=output_video_path,
    fps=get_video_fps(video_path),
    zone_margin=8.0,
    confirm_frames=3,
    exit_confirm_frames=3,
)

intrusion_events


## Release the inference session and GPU resources

Run this cell after finishing the notebook.


In [ ]:
_ = predictor.handle_request(
    request={
        "type": "close_session",
        "session_id": session_id,
    }
)
predictor.shutdown()
